# Project 2: Predicting HDB Resale Prices
## Notebook 2 - Modelling and Conclusion

**Objective:** To find out the best model to use for predicting HDB resale price based on the data set that has been cleaned in Notebook 1. Based on the result findings, recommendations will be given to address the problem statement and for further improvement.

1. [Libraries and Data Imports](#1-libraries-and-data-imports)
2. [Model Preparation](#model-preparation)
3. [Modelling](#modelling)
4. [Recommendations and Conclusion](#conclusion-and-recommendations)

## 1. Libraries and Data Imports

In [1]:
# Library Imports:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.linear_model import LinearRegression 
from sklearn import metrics
from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('fivethirtyeight')

%config InlineBackend.figure_format = 'retina'
%matplotlib inline

In [2]:
# Data Import:
model = pd.read_csv('../data/hdb_modelling_2.csv')

### Final Data Check

In [3]:
# One last look to confirm that the data is ready for model preparation:
def data_summary(df):
  with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(f'Shape(row,col): {df.shape}')
    print('-'*60)
    print(f'Number of duplicates: {df.duplicated().sum()}')
    print('-'*60)
    print('Columns containing missing values:')
    isnull_df = pd.DataFrame(df.isnull().sum()).reset_index()
    isnull_df.columns = ['col','num_nulls']
    isnull_df['null_pct'] = (isnull_df['num_nulls']/len(df)).round(2)
    df_nullcheck = isnull_df[isnull_df['num_nulls']>0]
    if len(df_nullcheck) == 0:
      print('None')
    else:
      print(df_nullcheck)
    print('-'*60)

data_summary(model)

Shape(row,col): (150634, 52)
------------------------------------------------------------
Number of duplicates: 0
------------------------------------------------------------
Columns containing missing values:
None
------------------------------------------------------------


In [4]:
# List number of unique elements in each column by data type:
print (f'Number of unique values in non-numerical columns:')
print('-'*60)
for col in model.columns:
    if model[col].dtype == 'object':
        print(f'{col}: {model[col].nunique()}')

print('-'*60)

print (f'Number of unique values in numerical columns:')
print('-'*60)
for col in model.columns:
    if model[col].dtype == 'int64' or model[col].dtype == 'float64':
        print(f'{col}: {model[col].nunique()}') 

Number of unique values in non-numerical columns:
------------------------------------------------------------
town: 26
street_name: 553
address: 9157
flat_model: 20
flat_type: 7
mrt_name: 94
bus_stop_name: 1657
pri_sch_name: 177
sec_sch_name: 134
flat_position: 3
------------------------------------------------------------
Number of unique values in numerical columns:
------------------------------------------------------------
id: 150634
resale_price: 3295
tranc_year: 10
tranc_month: 12
floor_area_sqft: 169
flat_floor: 22
total_floor: 41
hdb_age: 54
have_shops: 2
have_multi_carpark: 2
have_pavilion: 2
total_dwelling_units: 320
2room_sold: 99
5room_sold: 133
exec_sold: 103
multigen_sold: 7
studio_apartment_sold: 61
1room_rental: 11
3room_rental: 15
mall_nearest_distance: 8983
mall_within_500m: 7
mall_within_1km: 16
mall_within_2km: 42
hawker_nearest_distance: 9126
hawker_within_500m: 6
hawker_within_1km: 10
hawker_within_2km: 20
hawker_food_stalls: 52
hawker_market_stalls: 70
mrt_near

**Comments**
* As 'address' has many unique values, it will be excluded in the modelling to optimize computational time
* 'ID' is not required for modelling as it is not a numerical variable but for reference purpose

## 2. Model Preparation

### Train-Test Split

In [5]:
# Split data into X and y:
X = model.drop(columns=['resale_price', 'address', 'id'], axis=1)
y = model['resale_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

for dataset in [X_train, X_test, y_train, y_test]:
 print(dataset.shape)

(112975, 49)
(37659, 49)
(112975,)
(37659,)


In [6]:
#dtypes of X_train:
X_train.dtypes

tranc_year                     int64
tranc_month                    int64
town                          object
street_name                   object
floor_area_sqft              float64
flat_model                    object
flat_type                     object
flat_floor                     int64
total_floor                    int64
hdb_age                        int64
have_shops                     int64
have_multi_carpark             int64
have_pavilion                  int64
total_dwelling_units           int64
2room_sold                     int64
5room_sold                     int64
exec_sold                      int64
multigen_sold                  int64
studio_apartment_sold          int64
1room_rental                   int64
3room_rental                   int64
mall_nearest_distance        float64
mall_within_500m             float64
mall_within_1km              float64
mall_within_2km              float64
hawker_nearest_distance      float64
hawker_within_500m           float64
h

In [7]:
# Split X_train into categorical and numerical columns for respective scaling:
X_train_cat = X_train.select_dtypes(include='object')
X_train_num = X_train.select_dtypes(include=['int64', 'float64'])

print(f'X_train_cat shape: {X_train_cat.shape}')
print(f'X_train_num shape: {X_train_num.shape}')

X_test_cat = X_test.select_dtypes(include='object')
X_test_num = X_test.select_dtypes(include=['int64', 'float64'])

print(f'X_test_cat shape: {X_test_cat.shape}')
print(f'X_test_num shape: {X_test_num.shape}')

X_train_cat shape: (112975, 9)
X_train_num shape: (112975, 40)
X_test_cat shape: (37659, 9)
X_test_num shape: (37659, 40)


### Scaling

In [8]:
# One Hot Encoding for categorical variables:
enc = OneHotEncoder(drop='first', handle_unknown='ignore')
X_train_cat_enc = enc.fit_transform(X_train_cat)
X_test_cat_enc = enc.transform(X_test_cat)

In [9]:
# Assign encoded columns back to a dense matrix and assign to a dataframe:
enc_col_names = list(enc.get_feature_names_out()) # get the column names
X_train_cat_enc = pd.DataFrame(X_train_cat_enc.toarray(), columns=enc_col_names) 
X_test_cat_enc = pd.DataFrame(X_test_cat_enc.toarray(), columns=enc_col_names) 

In [10]:
#check if there are any Nan values in the encoded columns:
print(f'Number of NaN values in X_train_cat_enc: {X_train_cat_enc.isnull().sum().sum()}')
print(f'Number of NaN values in X_test_cat_enc: {X_test_cat_enc.isnull().sum().sum()}')


Number of NaN values in X_train_cat_enc: 0
Number of NaN values in X_test_cat_enc: 0


In [11]:
print(X_train_cat_enc.shape)
print(X_test_cat_enc.shape)

(112975, 2661)
(37659, 2661)


In [12]:
# Standard Scaler for numerical variables:
ss = StandardScaler()
X_train_num_ss = ss.fit_transform(X_train_num)
X_test_num_ss = ss.transform(X_test_num)

In [13]:
# Assign scaled columns back to a dataframe:
ss_col_names = list(ss.get_feature_names_out())
X_train_num_ss = pd.DataFrame(X_train_num_ss, columns=ss_col_names)
X_test_num_ss = pd.DataFrame(X_test_num_ss, columns=ss_col_names)

In [14]:
#check if there are any Nan values in the scaled columns:
print(f'Number of NaN values in X_train_num_ss: {X_train_num_ss.isnull().sum().sum()}')
print(f'Number of NaN values in X_test_num_ss: {X_test_num_ss.isnull().sum().sum()}')

Number of NaN values in X_train_num_ss: 0
Number of NaN values in X_test_num_ss: 0


In [15]:
print(X_train_num_ss.shape)
print(X_test_num_ss.shape)

(112975, 40)
(37659, 40)


In [16]:
# Merge the numerical and categorical variables back together:
X_train = pd.concat(objs=[X_train_num_ss, X_train_cat_enc], axis=1)
X_test = pd.concat(objs=[X_test_num_ss, X_test_cat_enc], axis=1)

# Check the shape of the processed data
print(f'Shape of X_train: {X_train.shape}')
print(f'Shape of X_test: {X_test.shape}')

Shape of X_train: (112975, 2701)
Shape of X_test: (37659, 2701)


In [17]:
#check if there are any Nan values in the processed data:
print(f'Number of NaN values in X_train: {X_train.isnull().sum().sum()}')
print(f'Number of NaN values in X_test: {X_test.isnull().sum().sum()}')

Number of NaN values in X_train: 0
Number of NaN values in X_test: 0


## 3. Modelling

### Linear Regression - Baseline Model

In [18]:
%%time
# Instantiate Linear Regression model:
lr = LinearRegression()

# Fit the model:
lr.fit(X_train, y_train)

CPU times: user 5min 5s, sys: 55.7 s, total: 6min 1s
Wall time: 4min 18s


LinearRegression()

### Baseline Model Performance

In [19]:
%%time
# R2 score
print(f'Train: {lr.score(X_train, y_train)}')
print(f'Test: {lr.score(X_test, y_test)}')

Train: 0.9355728396497345
Test: 0.9316923576476528
CPU times: user 2.79 s, sys: 2.87 s, total: 5.66 s
Wall time: 7.45 s


In [20]:
%%time
# Cross validation score:
cross_val_score(lr, X_train, y_train).mean()

CPU times: user 21min 39s, sys: 1min 18s, total: 22min 58s
Wall time: 9min 59s


-3524947720652819.5

In [21]:
# RMSE train score
y_train_pred = lr.predict(X_train)
print (f'train score: {np.sqrt(metrics.mean_squared_error(y_train, y_train_pred))}')


# RMSE test score
y_test_pred = lr.predict(X_test)
print (f'test score: {np.sqrt(metrics.mean_squared_error(y_train, y_train_pred))}')

train score: 36421.40036781122
test score: 36421.40036781122


### Ridge CV

In [22]:
%%time

# Instantiate Ridge regression model:
ridge = RidgeCV()

# Fit the model
ridge.fit(X_train,y_train)

CPU times: user 11min 15s, sys: 1min 10s, total: 12min 26s
Wall time: 8min 56s


RidgeCV()

In [23]:
# Find the optimal alpha for Ridge:
best_alpha = ridge.alpha_
print(f'Best alpha: {best_alpha}')

Best alpha: 0.1


### Ridge Model Performance

In [24]:
# Ridge R2 score
print(f'Train score: {ridge.score(X_train, y_train)}')
print(f'Test score: {ridge.score(X_test, y_test)}')

Train score: 0.9355549680271961
Test score: 0.931729958021917


In [25]:
# Cross validation score:
Ridge = Ridge(alpha = best_alpha)
cross_val_score(Ridge, X_train, y_train).mean()

0.9319556237826172

In [26]:
# RMSE train score
y_train_pred = ridge.predict(X_train)
print (f'train score: {np.sqrt(metrics.mean_squared_error(y_train, y_train_pred))}')


# RMSE test score
y_test_pred = ridge.predict(X_test)
print (f'test score: {np.sqrt(metrics.mean_squared_error(y_train, y_train_pred))}')

train score: 36426.45153243678
test score: 36426.45153243678


**Comments**

Ridge appears to perform slightly better than the baseline linear model. The cross_val_score is also significantly better than baseline, which indicates higher accuracy level of predictive accuracy on the training data set.

### Lasso Regression

In [27]:
%%time
lasso = LassoCV()
lasso.fit(X_train, y_train)

CPU times: user 8min 5s, sys: 40.2 s, total: 8min 45s
Wall time: 4min 48s


LassoCV()

In [28]:
# Find the optimal alpha value for Lasso:
best_alpha_lasso = lasso.alpha_
print(f'Optimal alpha value: {best_alpha_lasso}')

Optimal alpha value: 93.70168342510763


### Lasso Model Performance

In [29]:
# Lasso R2 Scores
print(f'Train score: {lasso.score(X_train, y_train)}')
print(f'Test score: {lasso.score(X_test, y_test)}')

Train score: 0.9003797016695468
Test score: 0.8998425889650962


In [30]:
# Cross validation score:
Lasso = Lasso(alpha = best_alpha_lasso)
cross_val_score(Lasso, X_train, y_train).mean()

0.8999464018842309

In [31]:
# RMSE train score
y_train_pred = lasso.predict(X_train)
print (f'train score: {np.sqrt(metrics.mean_squared_error(y_train, y_train_pred))}')


# RMSE test score
y_test_pred = lasso.predict(X_test)
print (f'test score: {np.sqrt(metrics.mean_squared_error(y_train, y_train_pred))}')

train score: 45289.34752462007
test score: 45289.34752462007


**Summary**

| Model           | Linear    | Ridge   | Lasso   |
| --------------- | --------- | ------- | ------- |
| R2 Train Score  | 0.93557   | 0.93555 | 0.90038 |
| R2 Test Score   | 0.93169   | 0.93173 | 0.89984 |
| Cross Val Score |  -3.5E+15 | 0.93196 | 0.89995 |
| RMSE Train      | 36421.4   | 36426.5 | 45289.3 |
| RMSE Test       | 36421.4   | 36426.5 | 45289.3 |

Ridge still has the best performance among the 3 models. 

## 4. Recommendations and Conclusion

### Model Coefficients

In [33]:
# Get ridge coefficients
ridge_coefs = pd.DataFrame({'variable':X_train.columns, 'coef':ridge.coef_, 'abs_coef':np.abs(ridge.coef_)})

ridge_coefs.sort_values('abs_coef', inplace=True, ascending=False)

ridge_coefs.head(20)

,variable,coef,abs_coef
633,flat_model_Terrace,362886.764354,362886.764354
341,street_name_KIM CHENG ST,211147.851532,211147.851532
380,street_name_MOH GUAN TER,209208.012182,209208.012182
2198,bus_stop_name_Opp River Valley High Sch,-206582.741695,206582.741695
319,street_name_JURONG WEST ST 52,194715.108003,194715.108003
352,street_name_LIM LIAK ST,187137.613275,187137.613275
344,street_name_KIM PONG RD,185053.748402,185053.748402
122,street_name_BOON LAY PL,177495.477331,177495.477331
2289,bus_stop_name_River Valley High Sch,-175477.018755,175477.018755
316,street_name_JURONG WEST ST 41,173940.244757,173940.244757


**Comments**

* As indicated above, the majority of the most significant features that have strong relationship with resale prices are mostly rare HDB flat type (two-storey terrace built in the 50s) and location related.
* Inferring from majority of the street names coefficients, flats located in **Tiong Bahru** are likely the most expensive due to its prime location (proximity to city, well-established amenities, travel convenience).
* Inferring from coefficients of Jurong West St 52/St 51/St 41, flats located **near to popular primary schools** (see map screenshot below). Rulang Primary School is known to be the top school in Jurong are also likely to be popular and hence cost more.

<img src="../images/Jurong West St 52.png" style="width: 50%; height: auto;">

* Inferring from coefficient of Tao Ching Road, flats near to **multiple good amenities** can be a key factor influencing price as well. Looking at a snapshot of its location on Google Map, flats located at this area are next to a huge garden and near to supermarket (NTUC) as well as hawker centre (Taman Jurong Food Centre).

<img src="../images/Tao Ching Road.png" style="width: 50%; height: auto;">

* Flats located in **Woodlands** relatively cost less due to its far north location and can be an option for buyers with tight budget.

**Conclusion**
* Best model to use for predicting resale prices = Ridge
* Key factors that have major influence on price = location, maturity and variety of amenities, schooling
* Valuable flat type is HDB Terrace and popular locations are Tiong Bahru and Jurong
* Woodlands can be a good option if one has tight budget

**Recommendations for Future Improvement**
* More research and analysis can be made to identify more accurately what features can be further eliminated to improve the model performance. 
* As many categorical features are non-quantitative in nature and may not have linear relationship with price, it might make more sense to use other more suitable models such as Random forest or Neural networks that can learn non-linear patterns better.